In [ ]:
import numpy as np
residuals = np.load('../../graph/COLIEE_data_quant.npy', allow_pickle=True)
print(residuals.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import torch


layerwise_data = list(zip(*residuals))
layerwise_arrays = [np.stack(layer, axis=0) for layer in layerwise_data]

# data = layerwise_arrays[3]
data = np.concatenate([emb for emb in layerwise_arrays], axis=1)
# data = np.sum([emb for emb in layerwise_arrays], axis=0)
print(data.shape)
sim = torch.tensor(cosine_similarity(data))
print(sim.shape)

In [ ]:
import json

with open('datasets/COLIEE/test/test_candidate_ids.json', 'r') as f:
    id_list = json.load(f)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

def generate_trec(np_arr, id_list, full_id_list, top_k=30, output_file="output.trec"):
    """
    np_arr:        Full embedding array (shape: N x D)
    id_list:       Subset of IDs to include in TREC (length M)
    full_id_list:  List mapping np_arr row index to original ID (length N)
    """
    id_to_index = {id_: i for i, id_ in enumerate(full_id_list)}
    selected_indices = [id_to_index[i] for i in id_list]

    filtered_arr = np_arr[selected_indices]

    cos_sim = cosine_similarity(filtered_arr)  # shape (M, M)
    print(cos_sim.shape)
    np.fill_diagonal(cos_sim, -np.inf)

    top_indices = np.argsort(cos_sim, axis=1)[:, -top_k:][:, ::-1]  # (M, top_k)

    with open(output_file, "w") as f:
        for i, query_id in enumerate(id_list):
            for rank, j in enumerate(top_indices[i]):
                doc_id = id_list[j]
                score = cos_sim[i, j]
                f.write(f"{query_id} Q0 {doc_id} {rank+1} \n")

    print(f"Written TREC file for {len(id_list)} samples → {output_file}")

generate_trec(data, id_list, list(range(len(data))), top_k=30, output_file="COLIEE.trec")
print(data.shape)